# C5 — Final XGBoost Multi-Objective Optimization

This version is designed specifically to improve the four probability metrics
without using an arbitrary sum such as:

`PR-AUC + ROC-AUC - Log Loss - Brier`

### Objectives

- **PR-AUC:** maximize
- **ROC-AUC:** maximize
- **Log Loss:** minimize
- **Brier Score:** minimize

Optuna uses genuine multi-objective optimization and returns a Pareto frontier.

### Leakage rules

- Hyperparameter tuning uses only C4 `train.csv`
- CV = 5-fold `StratifiedGroupKFold`
- Group = `case_id`
- Validation/test are untouched during tuning
- Preprocessing is fitted inside each CV fold
- Final threshold/calibration analysis is separate from hyperparameter tuning

### Efficiency

Each CV fold uses XGBoost early stopping. The median best iteration from the
CV folds is used for the final full-training fit.

In [1]:

# ============================================================
# 0. SETUP + LOAD C4 SPLITS
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.isotonic import IsotonicRegression

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data" / "c4_splits").exists():
    ROOT = ROOT.parent

SPLIT_DIR = ROOT / "data" / "c4_splits"
REPORT_DIR = ROOT / "reports" / "c5_final_multimetric"
MODEL_DIR = ROOT / "models" / "c5"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260824
TARGET = "y"
GROUP_COL = "case_id"

N_SPLITS = 5
N_TRIALS = 40
EARLY_STOPPING_ROUNDS = 60

train = pd.read_csv(SPLIT_DIR / "train.csv", low_memory=False)
valid = pd.read_csv(SPLIT_DIR / "validation.csv", low_memory=False)
test = pd.read_csv(SPLIT_DIR / "test.csv", low_memory=False)

for name, df in [("train", train), ("validation", valid), ("test", test)]:
    if TARGET not in df.columns or GROUP_COL not in df.columns:
        raise ValueError(f"{name} missing required columns.")
    y = pd.to_numeric(df[TARGET], errors="coerce")
    if y.isna().any() or not y.isin([0, 1]).all():
        raise ValueError(f"{name} target must be binary 0/1.")

print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)
print("Positive rates:",
      train[TARGET].mean(),
      valid[TARGET].mean(),
      test[TARGET].mean())


Train: (110385, 37)
Validation: (23471, 37)
Test: (23567, 37)
Positive rates: 0.2282194138696381 0.22994333432746794 0.22590911019646115


## 1. Feature Contract

Use the fixed C3/C4 model contract. Do not add identifiers, logging policy, or
post-decision leakage.

In [2]:

ID_COLUMNS = {
    "customer_id",
    "subscription_id",
    "case_id",
}

POLICY_COLUMNS = {
    "logging_policy",
}

INTERVENTION_COLUMNS = {
    "action_type",
    "channel",
}

FEATURES = [
    c for c in train.columns
    if c not in ID_COLUMNS | POLICY_COLUMNS | {TARGET}
]

CATEGORICAL_COLS = [
    c for c in FEATURES
    if (
        c in INTERVENTION_COLUMNS
        or not pd.api.types.is_numeric_dtype(train[c])
    )
]

NUMERIC_COLS = [
    c for c in FEATURES
    if (
        c not in CATEGORICAL_COLS
        and pd.api.types.is_numeric_dtype(train[c])
    )
]

print("Predictive features:", len(FEATURES))
print("Numeric:", len(NUMERIC_COLS))
print("Categorical:", len(CATEGORICAL_COLS))
print("Interventions:",
      [c for c in INTERVENTION_COLUMNS if c in train.columns])

assert not (set(FEATURES) & ID_COLUMNS)
assert not (set(FEATURES) & POLICY_COLUMNS)


Predictive features: 32
Numeric: 20
Categorical: 12
Interventions: ['action_type', 'channel']


## 2. Fold-Safe Preprocessing

In [3]:

def make_preprocessor():
    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
        ),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, NUMERIC_COLS),
            ("categorical", categorical_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop",
    )


## 3. Four Metrics

In [4]:

def evaluate(y_true, proba):
    proba = np.clip(
        np.asarray(proba, dtype=float),
        1e-7,
        1 - 1e-7,
    )
    y_true = np.asarray(y_true)

    return {
        "pr_auc": float(
            average_precision_score(y_true, proba)
        ),
        "roc_auc": float(
            roc_auc_score(y_true, proba)
        ),
        "log_loss": float(
            log_loss(y_true, proba)
        ),
        "brier": float(
            brier_score_loss(y_true, proba)
        ),
    }


## 4. XGBoost Search Space

The ranges are centered around the region already shown to work well, but
include enough room to discover a better bias/variance trade-off.

In [5]:

def xgb_params(trial):
    return {
        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.008, 0.08, log=True
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 40
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.70, 1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.70, 1.00
        ),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 3.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-5, 5.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.05, 30.0, log=True
        ),
    }


## 5. Multi-Objective Optuna

This is the key correction.

We do NOT invent a composite equation with incompatible raw scales.

Optuna directly optimizes:

1. maximize PR-AUC
2. maximize ROC-AUC
3. minimize Log Loss
4. minimize Brier Score

A trial is Pareto-optimal when no other trial is better on all four objectives.

In [8]:

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

X_train = train[FEATURES]
y_train = train[TARGET].astype(int)
groups = train[GROUP_COL]
def objective(trial):
    params = xgb_params(trial)

    fold_results = []
    best_iterations = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_train,
            y_train,
            groups,
        )
    ):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]

        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        prep = make_preprocessor()

        X_tr_enc = prep.fit_transform(X_tr)
        X_va_enc = prep.transform(X_va)

        model = XGBClassifier(
            **params,
            n_estimators=1500,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=SEED + fold,
            n_jobs=-1,
        )

        model.fit(
            X_tr_enc,
            y_tr,
            eval_set=[(X_va_enc, y_va)],
            verbose=False,
        )

        best_iter = getattr(
            model,
            "best_iteration",
            model.n_estimators - 1,
        )

        best_iterations.append(
            int(best_iter) + 1
        )

        proba = model.predict_proba(
            X_va_enc
        )[:, 1]

        fold_results.append(
            evaluate(
                y_va,
                proba,
            )
        )

    mean_metrics = {
        metric: float(
            np.mean(
                [
                    result[metric]
                    for result in fold_results
                ]
            )
        )
        for metric in [
            "pr_auc",
            "roc_auc",
            "log_loss",
            "brier",
        ]
    }

    trial.set_user_attr(
        "cv_pr_auc",
        mean_metrics["pr_auc"],
    )

    trial.set_user_attr(
        "cv_roc_auc",
        mean_metrics["roc_auc"],
    )

    trial.set_user_attr(
        "cv_log_loss",
        mean_metrics["log_loss"],
    )

    trial.set_user_attr(
        "cv_brier",
        mean_metrics["brier"],
    )

    trial.set_user_attr(
        "median_best_iteration",
        int(
            np.median(
                best_iterations
            )
        ),
    )

    # Multi-objective return.
    #
    # 1. maximize PR-AUC
    # 2. maximize ROC-AUC
    # 3. minimize Log Loss
    # 4. minimize Brier Score

    return (
        mean_metrics["pr_auc"],
        mean_metrics["roc_auc"],
        mean_metrics["log_loss"],
        mean_metrics["brier"],
    )

## 6. Run Multi-Objective Optuna

In [9]:
study = optuna.create_study(
    directions=[
        "maximize",   # PR-AUC
        "maximize",   # ROC-AUC
        "minimize",   # Log Loss
        "minimize",   # Brier
    ],
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(
    "Completed trials:",
    len(study.trials),
)

print(
    "Pareto-optimal trials:",
    len(study.best_trials),
)

  0%|          | 0/40 [00:00<?, ?it/s]

Completed trials: 40
Pareto-optimal trials: 3


## 7. Inspect the Pareto Frontier

There is no single mathematically correct winner when improving four different
metrics simultaneously.

The Pareto frontier contains the non-dominated candidate configurations.

In [10]:

pareto_rows = []

for trial in study.best_trials:
    pareto_rows.append({
        "trial": trial.number,
        "pr_auc": trial.values[0],
        "roc_auc": trial.values[1],
        "log_loss": trial.values[2],
        "brier": trial.values[3],
        "median_best_iteration":
            trial.user_attrs.get(
                "median_best_iteration"
            ),
    })

pareto_df = (
    pd.DataFrame(pareto_rows)
    .sort_values(
        "pr_auc",
        ascending=False,
    )
)

display(pareto_df)


,trial,pr_auc,roc_auc,log_loss,brier,median_best_iteration
2,36,0.812266,0.932599,0.267059,0.083358,1500
1,34,0.812219,0.932533,0.267012,0.083334,1500
0,21,0.811009,0.932567,0.266924,0.083443,1500


## 8. Choose a Candidate Using Validation — Not Test

We now evaluate the Pareto candidates on the untouched C4 validation set.

Selection rule:

1. Prefer the highest validation PR-AUC.
2. Among close PR-AUC candidates, prefer better Log Loss and Brier.
3. Do not use TEST to make this choice.

This is a model-selection step, so it is allowed to use VALIDATION.

In [11]:

candidate_rows = []

for trial in study.best_trials:
    params = trial.params.copy()

    n_estimators = trial.user_attrs.get(
        "median_best_iteration",
        500,
    )

    model = XGBClassifier(
        **params,
        n_estimators=int(n_estimators),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
    )

    prep = make_preprocessor()

    Xtr_candidate = prep.fit_transform(
        train[FEATURES]
    )
    Xva_candidate = prep.transform(
        valid[FEATURES]
    )

    model.fit(
        Xtr_candidate,
        train[TARGET].astype(int),
    )

    val_proba = model.predict_proba(
        Xva_candidate
    )[:, 1]

    m = evaluate(
        valid[TARGET],
        val_proba,
    )

    candidate_rows.append({
        "trial": trial.number,
        **m,
        "n_estimators": int(n_estimators),
    })

validation_candidates = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        [
            "pr_auc",
            "log_loss",
            "brier",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)

display(
    validation_candidates
)


,trial,pr_auc,roc_auc,log_loss,brier,n_estimators
1,34,0.811842,0.934672,0.265189,0.083017,1500
2,36,0.811625,0.934149,0.265970,0.083120,1500
0,21,0.811146,0.934699,0.265090,0.083116,1500


## 9. Freeze the Best Validation Candidate

This candidate is selected using validation only.

In [12]:

selected_trial_number = int(
    validation_candidates.iloc[0]["trial"]
)

selected_trial = next(
    trial
    for trial in study.best_trials
    if trial.number == selected_trial_number
)

selected_params = selected_trial.params.copy()

selected_n_estimators = int(
    selected_trial.user_attrs.get(
        "median_best_iteration",
        500,
    )
)

print(
    "Selected trial:",
    selected_trial_number,
)

print(
    "Selected validation metrics:"
)

print(
    validation_candidates.iloc[0]
)

print(
    "\nSelected parameters:"
)

print(
    selected_params
)

print(
    "\nSelected n_estimators:",
    selected_n_estimators,
)


Selected trial: 34
Selected validation metrics:
trial             34.000000
pr_auc             0.811842
roc_auc            0.934672
log_loss           0.265189
brier              0.083017
n_estimators    1500.000000
Name: 1, dtype: float64

Selected parameters:
{'max_depth': 10, 'learning_rate': 0.021459711073767162, 'min_child_weight': 3, 'subsample': 0.914399817887789, 'colsample_bytree': 0.9433904720436755, 'gamma': 2.1849656090391902, 'reg_alpha': 1.8179248677042483, 'reg_lambda': 2.806510548553804}

Selected n_estimators: 1500


## 10. Final Model on Complete TRAIN

Now fit the frozen candidate on the entire C4 TRAIN set.

In [13]:

final_prep = make_preprocessor()

Xtr = final_prep.fit_transform(
    train[FEATURES]
)
Xva = final_prep.transform(
    valid[FEATURES]
)
Xte = final_prep.transform(
    test[FEATURES]
)

final_model = XGBClassifier(
    **selected_params,
    n_estimators=selected_n_estimators,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

final_model.fit(
    Xtr,
    train[TARGET].astype(int),
)

train_proba = final_model.predict_proba(
    Xtr
)[:, 1]

valid_proba = final_model.predict_proba(
    Xva
)[:, 1]

test_proba = final_model.predict_proba(
    Xte
)[:, 1]

final_results = pd.DataFrame([
    {
        "split": "train",
        **evaluate(
            train[TARGET],
            train_proba,
        ),
    },
    {
        "split": "validation",
        **evaluate(
            valid[TARGET],
            valid_proba,
        ),
    },
    {
        "split": "test",
        **evaluate(
            test[TARGET],
            test_proba,
        ),
    },
])

display(final_results)


,split,pr_auc,roc_auc,log_loss,brier
0,train,0.912605,0.969082,0.201912,0.060304
1,validation,0.811842,0.934672,0.265189,0.083017
2,test,0.827829,0.938963,0.254147,0.078582


## 11. OOF Calibration for Log Loss / Brier

This calibration stage uses only out-of-fold predictions from TRAIN.

It is intended mainly to improve probability quality.

We will compare before vs after calibration and keep the calibrated version only
if it improves validation probability quality without unacceptable ranking loss.

In [14]:

oof_proba = np.zeros(
    len(train),
    dtype=float,
)

for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        train[FEATURES],
        train[TARGET].astype(int),
        train[GROUP_COL],
    )
):
    fold_prep = make_preprocessor()

    X_fold_tr = fold_prep.fit_transform(
        train[FEATURES].iloc[tr_idx]
    )
    X_fold_va = fold_prep.transform(
        train[FEATURES].iloc[va_idx]
    )

    fold_model = XGBClassifier(
        **selected_params,
        n_estimators=selected_n_estimators,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED + 100 + fold,
        n_jobs=-1,
    )

    fold_model.fit(
        X_fold_tr,
        train[TARGET].iloc[tr_idx].astype(int),
    )

    oof_proba[va_idx] = fold_model.predict_proba(
        X_fold_va
    )[:, 1]

calibrator = IsotonicRegression(
    y_min=1e-6,
    y_max=1 - 1e-6,
    out_of_bounds="clip",
)

calibrator.fit(
    oof_proba,
    train[TARGET].astype(int).to_numpy(),
)

valid_proba_cal = calibrator.predict(
    valid_proba
)

test_proba_cal = calibrator.predict(
    test_proba
)

calibration_results = pd.DataFrame([
    {
        "split": "validation",
        "state": "uncalibrated",
        **evaluate(
            valid[TARGET],
            valid_proba,
        ),
    },
    {
        "split": "validation",
        "state": "calibrated",
        **evaluate(
            valid[TARGET],
            valid_proba_cal,
        ),
    },
    {
        "split": "test",
        "state": "uncalibrated",
        **evaluate(
            test[TARGET],
            test_proba,
        ),
    },
    {
        "split": "test",
        "state": "calibrated",
        **evaluate(
            test[TARGET],
            test_proba_cal,
        ),
    },
])

display(
    calibration_results
)


,split,state,pr_auc,roc_auc,log_loss,brier
0,validation,uncalibrated,0.811842,0.934672,0.265189,0.083017
1,validation,calibrated,0.806643,0.934496,0.263491,0.083046
2,test,uncalibrated,0.827829,0.938963,0.254147,0.078582
3,test,calibrated,0.820949,0.938839,0.253238,0.078626


## 12. Compare Against Existing C5 Baselines

In [15]:

baseline_path = (
    ROOT
    / "reports"
    / "c5_baseline"
    / "baseline_metrics.csv"
)

if baseline_path.exists():

    baseline = pd.read_csv(
        baseline_path
    )

    display(
        baseline[
            [
                "model",
                "split",
                "pr_auc",
                "roc_auc",
                "log_loss",
                "brier",
            ]
        ]
        .sort_values(
            [
                "split",
                "pr_auc",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

else:
    print("Original C5 baseline report not found.")


,model,split,pr_auc,roc_auc,log_loss,brier
5,gradient_boosting,test,0.787352,0.914982,0.300595,0.090900
3,logistic_regression,test,0.723009,0.881096,0.341749,0.104604
1,base_rate,test,0.225909,0.500000,0.534301,0.174880
4,gradient_boosting,validation,0.774632,0.910991,0.310199,0.094811
2,logistic_regression,validation,0.710845,0.872573,0.353747,0.109135
0,base_rate,validation,0.229943,0.500000,0.539216,0.177072


## 13. Save Results

In [16]:

study.trials_dataframe().to_csv(
    REPORT_DIR / "optuna_all_trials.csv",
    index=False,
)

pareto_df.to_csv(
    REPORT_DIR / "optuna_pareto_frontier.csv",
    index=False,
)

validation_candidates.to_csv(
    REPORT_DIR / "validation_candidate_comparison.csv",
    index=False,
)

final_results.to_csv(
    REPORT_DIR / "final_results_uncalibrated.csv",
    index=False,
)

calibration_results.to_csv(
    REPORT_DIR / "calibration_comparison.csv",
    index=False,
)

with open(
    REPORT_DIR / "selected_model.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "optimization": {
                "type": "multi_objective",
                "objectives": [
                    "maximize_pr_auc",
                    "maximize_roc_auc",
                    "minimize_log_loss",
                    "minimize_brier",
                ],
            },
            "selection": {
                "selected_trial": selected_trial_number,
                "rule": (
                    "highest validation PR-AUC among "
                    "Pareto-optimal candidates, with "
                    "Log Loss and Brier as tie-breakers"
                ),
            },
            "parameters": selected_params,
            "n_estimators": selected_n_estimators,
        },
        f,
        indent=2,
    )

inference_bundle = {
    "preprocessor": final_prep,
    "model": final_model,
    "calibrator": calibrator,
    "features": FEATURES,
}

joblib.dump(
    inference_bundle,
    MODEL_DIR / "xgboost_c5_final_multimetric.joblib",
)

print(
    "C5 final multi-objective artifacts saved."
)


C5 final multi-objective artifacts saved.
